# Polaris API → SQL → MinIO Access Map (Phase 1)

**Purpose:** for every Polaris Management API and Iceberg REST Catalog API operation, record

1. the exact SQL issued against the PostgreSQL metastore, and which tables it touches,
2. the MinIO objects/paths it accesses,
3. whether any statement lacks a selective index — the headline question.

**LOCAL ONLY.** This notebook enables debug logging, seeds ~16,000 entities and tears them down.
It must never run against the shared company DEV or PROD realms.

**Run order:** every cell top-to-bottom (`Restart & Run All`). Section 2 (schema drift) runs
*before* anything else, because a missing index caused by an unmigrated local schema is a
different problem from one upstream never created.

**Before running:** complete `doc-instrumentation-runbook.md`. The preflight in section 1 fails
loudly if the log streams are not live — a silent miss would look like "this API issued no SQL",
which is exactly the wrong conclusion.

## 0. Bootstrap

In [ ]:
import gc
import json
import pathlib
import sys
from datetime import datetime

sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from polaris_test_utils import *  # noqa: F403 — init_env, root_token, PG_* globals

init_env("local")                     # default env; never "dev"/"prod" here
require_not_prod("api-sql-profile: enables debug logging and seeds ~16k entities")

from api_trace import (
    FileStream,
    Tracer,
    api_minio_matrix,
    api_table_matrix,
    records_to_rows,
    statement_inventory,
    unknown_tables,
)
from iceberg_rest import IcebergREST, build_create_table_payload, build_schema
from polaris_rest import PolarisREST
from polaris_seed import SeedSpec, find_strays, seed, teardown, verify_counts
from schema_audit import (
    analyze_tables,
    audit_statements,
    check_hypotheses,
    compare_schema,
    describe_schema,
    rank_statements,
    statistics_are_stale,
    table_stats,
)

# Second guard, independent of require_not_prod: this notebook is destructive.
assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — api-sql-profile is local-only."
)

TOKEN = root_token()
pc = PolarisREST(POLARIS_URL, REALM, token=TOKEN)
ic = IcebergREST(POLARIS_URL, REALM, token=TOKEN)

print(f"Polaris  : {POLARIS_URL}  realm={REALM}")
print(f"Postgres : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"Started  : {datetime.now():%Y-%m-%d %H:%M:%S}")

### Capture directory and stream wiring

`CAPTURE_DIR` holds the raw log captures. **It must be gitignored** — captures contain bound
parameters, and statements against `principal_authentication_data` carry secret hashes and salts.
`api_trace` redacts those before they reach any record or document, but the raw files themselves
are not safe to commit.

In [ ]:
CAPTURE_DIR = pathlib.Path("capture")
CAPTURE_DIR.mkdir(exist_ok=True)

POLARIS_LOG = CAPTURE_DIR / "polaris.log"
PG_LOG = CAPTURE_DIR / "pg.log"
MINIO_TRACE = CAPTURE_DIR / "minio.json"

# Direct-to-Postgres for the mapping run (decision #6): bypassing PgBouncer removes
# the session-attribution ambiguity. The pooler's own cost is measured in a later phase.
import psycopg2

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
)
conn.autocommit = True

gitignore = CAPTURE_DIR / ".gitignore"
if not gitignore.exists():
    gitignore.write_text("*\n", encoding="utf-8")   # never commit raw captures
print(f"capture dir: {CAPTURE_DIR.resolve()} (gitignored)")

## 1. Instrumentation preflight

In [ ]:
def preflight():
    """Verify each stream is actually producing output before the sweep starts.

    A stream that is configured but not running yields empty captures, which read
    as "this API touched no tables" — the most misleading possible failure. Fail
    here instead.
    """
    issues, warnings = [], []

    if not POLARIS_LOG.exists() or POLARIS_LOG.stat().st_size == 0:
        issues.append(f"{POLARIS_LOG} missing/empty — is the log tail running?")
    else:
        text = POLARIS_LOG.read_text(errors="replace")[-200_000:]
        if "DatasourceOperations" not in text:
            issues.append(
                "Polaris log has no DatasourceOperations lines. SQL DEBUG logging is "
                "off. Set QUARKUS_LOG_CATEGORY__ORG_APACHE_POLARIS_PERSISTENCE_"
                "RELATIONAL_JDBC__LEVEL=DEBUG and confirm quarkus.log.min-level "
                "allows DEBUG (build-time property — a runtime category setting is "
                "silently ignored if min-level is higher)."
            )

    if not PG_LOG.exists() or PG_LOG.stat().st_size == 0:
        warnings.append(
            f"{PG_LOG} missing/empty — proceeding WITHOUT server-side durations. "
            "Table mapping still works; timing columns will be empty."
        )
    if not MINIO_TRACE.exists() or MINIO_TRACE.stat().st_size == 0:
        warnings.append(
            f"{MINIO_TRACE} missing/empty — proceeding WITHOUT MinIO attribution. "
            "The 'which storage path' question cannot be answered from this run."
        )

    for w in warnings:
        print(f"WARN  {w}")
    if issues:
        raise RuntimeError("Preflight failed:\n  - " + "\n  - ".join(issues))
    print("Preflight OK")


preflight()

tracer = Tracer(
    polaris_log=FileStream(str(POLARIS_LOG)),
    pg_log=FileStream(str(PG_LOG)) if PG_LOG.exists() else None,
    minio_trace=FileStream(str(MINIO_TRACE)) if MINIO_TRACE.exists() else None,
    settle_s=0.30,   # logs flush asynchronously; without this the tail of a call is missed
)

## 2. Schema drift and index presence

**This runs first, deliberately.** A missing index has two very different causes:

* the running Polaris version's schema never defined it → an upstream gap
* this instance was created under an older schema and never migrated → a local migration gap

Only the drift check distinguishes them. Drawing a conclusion from an `EXPLAIN` before running
this risks reporting a local migration problem as an upstream bug.

In [ ]:
snapshot = describe_schema(conn)
drift = compare_schema(snapshot)

print(f"schema version : {drift['version_found']} (expected {drift['version_expected']})")
print(f"verdict        : {drift['verdict']}")
print(f"tables         : {snapshot['tables']}")
print(f"indexes        : {len(snapshot['indexes'])}")
for note in drift["notes"]:
    print(f"  note: {note}")

if drift["missing_indexes"]:
    print("\nMISSING EXPECTED ACCESS PATHS:")
    for i in drift["missing_indexes"]:
        print(f"  - {i['name']}  {i['table']}({', '.join(i['columns'])})  [{i['kind']}]")

if drift["extra_indexes"]:
    print("\nIndexes present but not in the expected v2 set (fine — often hand-added):")
    for i in drift["extra_indexes"]:
        print(f"  - {i['name']}  {i['table']}({', '.join(i['columns'])})")

In [ ]:
import pandas as pd

df_idx = pd.DataFrame(snapshot["indexes"])[["table", "name", "columns", "is_unique", "is_primary"]]
df_idx["columns"] = df_idx["columns"].apply(", ".join)
df_idx.sort_values(["table", "name"]).reset_index(drop=True)

## 3. Seed fixture — OPTIONAL, SLOW, DESTRUCTIVE

An `EXPLAIN` against a near-empty metastore proves nothing: PostgreSQL picks a sequential scan on
a small table because that is genuinely cheaper. The fixture exists so the planner's choice
carries information.

**Cost:** ~17,000–33,000 API calls (15–30 min), ~10,000 MinIO objects, ~16,000 `entities` rows and
~25,000 `grant_records` rows left in the realm until teardown (section 9).

Set `RUN_SEED = True` to build it. Leave it `False` to map the API surface against whatever data
already exists — the mapping is valid either way; only the *index verdicts* need the volume.

`SEED_TABLES = False` gives a metadata-only seed: ~10x faster, still fully populates
`grant_records`, which is the table the grantee-index hypothesis is about.

In [ ]:
RUN_SEED = False          # <-- set True to build the fixture
SEED_TABLES = True        # <-- False for a much faster metadata-only seed
LEDGER = str(CAPTURE_DIR / "seed_ledger.json")

spec = SeedSpec(n_users=1000, namespaces_per_catalog=2, tables_per_namespace=5,
                create_tables=SEED_TABLES)
print("projected:", json.dumps(spec.expected_counts(), indent=2))

if RUN_SEED:
    def on_progress(done, total, res):
        rate = res.calls / max(res.elapsed_s, 0.001)
        print(f"  {done}/{total} users  calls={res.calls}  "
              f"{rate:.0f} calls/s  elapsed={res.elapsed_s:.0f}s")

    result = seed(pc, ic, spec, ledger_path=LEDGER, bucket=BUCKET,
                  minio_endpoint=MINIO_ENDPOINT, on_progress=on_progress)
    print("\n" + result.summary())
    if result.invalid_privileges:
        print(
            "NOTE: the server rejected these privilege names — grant_records volume "
            f"is lower than projected: {sorted(set(result.invalid_privileges))}"
        )
    if result.failed_users:
        print(f"FAILED users (re-run this cell to retry): {result.failed_users[:5]}")
else:
    print("Seeding skipped. Index verdicts may come back TOO_SMALL.")

In [ ]:
verification = verify_counts(conn, spec) if RUN_SEED else None
if verification:
    print(json.dumps(verification["actual"], indent=2))
    for note in verification["notes"]:
        print(f"  note: {note}")
else:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM polaris_schema.entities")
        n_ent = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM polaris_schema.grant_records")
        n_grant = cur.fetchone()[0]
    print(f"current volume: entities={n_ent}  grant_records={n_grant}")
    if n_grant < 5000:
        print(
            "  note: grant_records is small — the grantee access-path hypothesis "
            "cannot be settled at this volume (verdicts will read TOO_SMALL)."
        )

## 4. Probe fixture

A small dedicated catalog for the sweep itself, separate from the bulk seed. Each traced operation
needs a known target, and mixing probe entities into the seeded set would make teardown ambiguous.

In [ ]:
PROBE = f"apiprofile{int(datetime.now().timestamp())}"
PROBE_CAT, PROBE_NS, PROBE_NS2 = f"{PROBE}_cat", "probe_ns", "probe_ns2"
PROBE_TBL, PROBE_VIEW = "probe_tbl", "probe_view"
PROBE_PRINCIPAL, PROBE_PROLE, PROBE_CROLE = f"{PROBE}_p", f"{PROBE}_pr", f"{PROBE}_cr"

schema = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

r = pc.create_catalog(PROBE_CAT, bucket=BUCKET, minio_endpoint=MINIO_ENDPOINT)
assert r.status_code < 300, r.text
for ns in (PROBE_NS, PROBE_NS2):
    assert ic.create_namespace(PROBE_CAT, ns).status_code < 300
assert ic.create_table(
    PROBE_CAT, PROBE_NS, build_create_table_payload(PROBE_TBL, schema)
).status_code < 300
print(f"probe fixture ready: {PROBE_CAT}")

## 5. The sweep

Full sweep of both API surfaces (decision #1), with the core CRUD subset marked `core=True` so the
reports can rank it first.

Each operation is one cold call inside a trace window. **Phase 1 measures mapping, not latency** —
the repeat/cache protocol is a later phase (decisions #3 and #7), so a single call per operation is
the right granularity here.

In [ ]:
def probe(api, method, path, fn, core=False):
    """Trace one API call. Failures are recorded and the sweep continues —
    a 403 or 404 still tells you which tables the authorization path touched."""
    try:
        with tracer.trace(api, method, path) as rec:
            resp = fn()
            rec.status = getattr(resp, "status_code", None)
    except Exception as exc:
        print(f"  ! {api}: {type(exc).__name__}: {exc}")
        return None
    rec.core = core
    flag = "*" if core else " "
    print(f"{flag} {api:<44} {rec.status}  sql={rec.sql_count:<3} "
          f"minio={rec.minio_count:<3} {rec.wall_ms:.0f}ms  {rec.cache_verdict}")
    return rec


print("core ops marked with *\n")
print("--- Iceberg REST: config & namespaces ---")
probe("iceberg.get_config", "GET", "/v1/config",
      lambda: ic.get_config(warehouse=PROBE_CAT), core=True)
probe("iceberg.list_namespaces", "GET", "/v1/{cat}/namespaces",
      lambda: ic.list_namespaces(PROBE_CAT), core=True)
probe("iceberg.load_namespace", "GET", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.load_namespace(PROBE_CAT, PROBE_NS), core=True)
probe("iceberg.head_namespace", "HEAD", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.head_namespace(PROBE_CAT, PROBE_NS))
probe("iceberg.update_namespace_properties", "POST", "/v1/{cat}/namespaces/{ns}/properties",
      lambda: ic.update_namespace_properties(PROBE_CAT, PROBE_NS, updates={"k": "v"}))
probe("iceberg.create_namespace", "POST", "/v1/{cat}/namespaces",
      lambda: ic.create_namespace(PROBE_CAT, "probe_ns_tmp"), core=True)
probe("iceberg.drop_namespace", "DELETE", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.drop_namespace(PROBE_CAT, "probe_ns_tmp"), core=True)

In [ ]:
print("--- Iceberg REST: tables ---")
probe("iceberg.list_tables", "GET", "/v1/{cat}/namespaces/{ns}/tables",
      lambda: ic.list_tables(PROBE_CAT, PROBE_NS), core=True)
probe("iceberg.load_table", "GET", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), core=True)
probe("iceberg.load_table[snapshots=refs]", "GET", "/v1/{cat}/.../tables/{tbl}?snapshots=refs",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL, snapshots="refs"))
probe("iceberg.head_table", "HEAD", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.head_table(PROBE_CAT, PROBE_NS, PROBE_TBL))
probe("iceberg.create_table", "POST", "/v1/{cat}/namespaces/{ns}/tables",
      lambda: ic.create_table(PROBE_CAT, PROBE_NS,
                              build_create_table_payload("probe_tbl2", schema)), core=True)
probe("iceberg.stage_create_table", "POST", "/v1/{cat}/namespaces/{ns}/tables[stage]",
      lambda: ic.stage_create_table(PROBE_CAT, PROBE_NS,
                                    build_create_table_payload("probe_staged", schema)))
probe("iceberg.commit_table", "POST", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.commit_table(PROBE_CAT, PROBE_NS, PROBE_TBL,
                              [{"action": "set-properties", "updates": {"p": "1"}}]), core=True)
probe("iceberg.rename_table", "POST", "/v1/{cat}/tables/rename",
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, "probe_tbl2", PROBE_NS, "probe_tbl3"))
probe("iceberg.report_metrics", "POST", "/v1/{cat}/.../tables/{tbl}/metrics",
      lambda: ic.report_metrics(PROBE_CAT, PROBE_NS, PROBE_TBL,
                                {"report-type": "scan-report", "table-name": PROBE_TBL}))
probe("iceberg.load_table[missing]", "GET", "/v1/{cat}/.../tables/{missing}",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, "does_not_exist"))
probe("iceberg.drop_table", "DELETE", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.drop_table(PROBE_CAT, PROBE_NS, "probe_tbl3"), core=True)

In [ ]:
print("--- Iceberg REST: views ---")
view_payload = {
    "name": PROBE_VIEW,
    "schema": schema,
    "view-version": {
        "version-id": 1, "schema-id": 0, "timestamp-ms": 0,
        "summary": {"operation": "create"},
        "representations": [{"type": "sql", "sql": "SELECT 1", "dialect": "spark"}],
        "default-namespace": [PROBE_NS],
    },
    "properties": {},
}
probe("iceberg.create_view", "POST", "/v1/{cat}/namespaces/{ns}/views",
      lambda: ic.create_view(PROBE_CAT, PROBE_NS, view_payload))
probe("iceberg.list_views", "GET", "/v1/{cat}/namespaces/{ns}/views",
      lambda: ic.list_views(PROBE_CAT, PROBE_NS))
probe("iceberg.load_view", "GET", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.load_view(PROBE_CAT, PROBE_NS, PROBE_VIEW))
probe("iceberg.head_view", "HEAD", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.head_view(PROBE_CAT, PROBE_NS, PROBE_VIEW))
probe("iceberg.rename_view", "POST", "/v1/{cat}/views/rename",
      lambda: ic.rename_view(PROBE_CAT, PROBE_NS, PROBE_VIEW, PROBE_NS, "probe_view2"))
probe("iceberg.drop_view", "DELETE", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.drop_view(PROBE_CAT, PROBE_NS, "probe_view2"))

In [ ]:
print("--- Management API: catalogs, principals, roles, grants ---")
probe("mgmt.list_catalogs", "GET", "/v1/catalogs", lambda: pc.list_catalogs(), core=True)
probe("mgmt.get_catalog", "GET", "/v1/catalogs/{cat}",
      lambda: pc.get_catalog(PROBE_CAT), core=True)
probe("mgmt.create_principal", "POST", "/v1/principals",
      lambda: pc.create_principal(PROBE_PRINCIPAL), core=True)
probe("mgmt.get_principal", "GET", "/v1/principals/{p}",
      lambda: pc.get_principal(PROBE_PRINCIPAL))
probe("mgmt.list_principals", "GET", "/v1/principals", lambda: pc.list_principals())
probe("mgmt.create_principal_role", "POST", "/v1/principal-roles",
      lambda: pc.create_principal_role(PROBE_PROLE))
probe("mgmt.get_principal_role", "GET", "/v1/principal-roles/{r}",
      lambda: pc.get_principal_role(PROBE_PROLE))
probe("mgmt.list_principal_roles", "GET", "/v1/principal-roles",
      lambda: pc.list_principal_roles())
probe("mgmt.assign_principal_role", "PUT", "/v1/principals/{p}/principal-roles",
      lambda: pc.assign_principal_role_to_principal(PROBE_PRINCIPAL, PROBE_PROLE))
probe("mgmt.create_catalog_role", "POST", "/v1/catalogs/{cat}/catalog-roles",
      lambda: pc.create_catalog_role(PROBE_CAT, PROBE_CROLE))
probe("mgmt.list_catalog_roles", "GET", "/v1/catalogs/{cat}/catalog-roles",
      lambda: pc.list_catalog_roles(PROBE_CAT))
probe("mgmt.assign_catalog_role", "PUT", "/v1/principal-roles/{r}/catalog-roles/{cat}",
      lambda: pc.assign_catalog_role_to_principal_role(PROBE_CAT, PROBE_PROLE, PROBE_CROLE))
probe("mgmt.grant_privilege", "PUT", "/v1/catalogs/{cat}/catalog-roles/{cr}/grants",
      lambda: pc.grant_privilege(PROBE_CAT, PROBE_CROLE, "TABLE_READ_DATA"), core=True)
probe("mgmt.list_grants", "GET", "/v1/catalogs/{cat}/catalog-roles/{cr}/grants",
      lambda: pc.list_grants(PROBE_CAT, PROBE_CROLE), core=True)
probe("mgmt.list_principals_for_principal_role", "GET",
      "/v1/principal-roles/{r}/principals",
      lambda: pc.list_principals_for_principal_role(PROBE_PROLE))
probe("mgmt.reset_principal_credentials", "POST", "/v1/principals/{p}/reset",
      lambda: pc.reset_principal_credentials(PROBE_PRINCIPAL))
probe("mgmt.delete_catalog_role", "DELETE", "/v1/catalogs/{cat}/catalog-roles/{cr}",
      lambda: pc.delete_catalog_role(PROBE_CAT, PROBE_CROLE))
probe("mgmt.delete_principal_role", "DELETE", "/v1/principal-roles/{r}",
      lambda: pc.delete_principal_role(PROBE_PROLE))
probe("mgmt.delete_principal", "DELETE", "/v1/principals/{p}",
      lambda: pc.delete_principal(PROBE_PRINCIPAL), core=True)

records = tracer.records
print(f"\ntraced {len(records)} operations")

## 6. Results — API → table and API → MinIO

In [ ]:
unknown = unknown_tables(records)
if unknown:
    print(
        f"WARNING: tables outside the expected schema-v2 set were observed: {unknown}\n"
        "Either the SQL parser mis-read a statement, or the deployed schema is not the "
        "version this audit assumes. Resolve before trusting the mapping below."
    )

df_summary = pd.DataFrame(records_to_rows(records))
df_summary["core"] = [getattr(r, "core", False) for r in records]
df_summary.sort_values(["core", "sql_count"], ascending=[False, False]).reset_index(drop=True)

In [ ]:
matrix = api_table_matrix(records)
all_tables = sorted({t for row in matrix.values() for t in row})
df_matrix = pd.DataFrame(
    [{"api": api, **{t: matrix[api].get(t, "") for t in all_tables}} for api in matrix]
).set_index("api")
print("R = read, W = write, RW = both\n")
df_matrix

In [ ]:
minio_matrix = api_minio_matrix(records)
rows = [
    {"api": api, "method": e["method"], "path": e["path"], "count": e["count"]}
    for api, entries in minio_matrix.items()
    for e in entries
]
if rows:
    display(pd.DataFrame(rows))
else:
    print(
        "No MinIO activity captured. Either the trace stream was not running "
        "(see the preflight warnings), or Polaris genuinely performed no object "
        "I/O for these operations — worth distinguishing before concluding either."
    )

## 7. Statement inventory and index audit

The distinct statement set, then an `EXPLAIN` verdict for each. Statements are ranked by
`total_ms` (mean × calls), **not** by max duration: the slowest single query is rarely the problem,
whereas a moderately slow query on the authorization path runs on every request.

In [ ]:
inventory = statement_inventory(records)
print(f"{len(inventory)} distinct statements\n")
pd.DataFrame(rank_statements(inventory, top=25))[
    ["table", "verb", "calls", "total_ms", "mean_ms", "max_ms", "sql"]
]

### Refresh planner statistics before EXPLAIN

`pg_class.reltuples` is only updated by VACUUM/ANALYZE, and this cluster throttles autovacuum
(`autovacuum_naptime = 30s`, `autovacuum_max_workers = 2`). Straight after a seed the planner still
believes the tables are empty, so it picks sequential scans that are correct *for the statistics it
has* and wrong for the data actually present — and the audit would record a false `SEQ_SCAN`.

This is not optional. Skipping it invalidates every index verdict below.

In [ ]:
stale = statistics_are_stale(conn)
if stale:
    print(f"statistics never gathered for: {stale}")
print("running ANALYZE ...")
print("analyzed:", analyze_tables(conn))

remaining = statistics_are_stale(conn)
assert not remaining, (
    f"still no statistics for {remaining} — EXPLAIN verdicts would be meaningless. "
    "Check permissions on these tables before continuing."
)
print("planner statistics are current")

In [ ]:
# Attach representative parameters so each statement can actually be EXPLAINed.
params_by_sql = {}
for rec in records:
    for s in rec.sql:
        from api_trace import normalize_sql
        key = normalize_sql(s.sql)
        if s.params and key not in params_by_sql:
            params_by_sql[key] = s.params
for entry in inventory:
    entry["params"] = params_by_sql.get(entry["sql"])

audit = audit_statements(conn, inventory)
df_audit = pd.DataFrame(audit)[
    ["verdict", "table", "verb", "calls", "table_rows", "plan_ms",
     "rows_removed_by_filter", "seq_scans", "index_scans", "detail", "sql"]
]
print(df_audit["verdict"].value_counts().to_string())
df_audit.sort_values("verdict").reset_index(drop=True)

In [ ]:
hyps = check_hypotheses(conn, audit)
for h in hyps:
    print(f"[{h['status']:<12}] {h['id']}  (severity: {h['severity']})")
    print(f"    method : {h['source_method']}")
    print(f"    claim  : {h['claim']}")
    print(f"    impact : {h['why_it_matters']}")
    print(f"    remedy : {h['remedy']}")
    for e in h["evidence"][:2]:
        print(f"    evidence: {str(e)[:120]}")
    print()

if any(h["status"] == "INCONCLUSIVE" for h in hyps):
    print(
        "INCONCLUSIVE results usually mean insufficient data volume. Set RUN_SEED = True "
        "in section 3 and re-run — a Seq Scan on a small table is the correct plan and "
        "proves nothing either way."
    )

In [ ]:
stats = table_stats(conn)
print("seq_scan vs idx_scan per table — a high seq_scan_ratio with large seq_tup_read")
print("is the smoking gun for a missing access path, and needs no per-query attribution.\n")
pd.DataFrame(stats)[
    ["table", "n_live_tup", "seq_scan", "seq_tup_read", "idx_scan",
     "seq_scan_ratio", "n_tup_upd", "n_tup_hot_upd", "hot_update_ratio"]
]

## 8. Write the reports

In [ ]:
def write_reports():
    """Emit doc-api-sql-matrix.md and doc-index-audit.md.

    All content routes through the redacting record objects, never raw log text,
    so no secret material can reach a committed document.
    """
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")

    lines = [
        "# API → SQL → MinIO Access Matrix",
        "",
        f"Generated {ts} from a live local run — Polaris {POLARIS_VERSION}, realm {REALM}.",
        f"Schema version {drift['version_found']} ({drift['verdict']}).",
        "",
        "R = read, W = write, RW = both.",
        "",
        "## API → PostgreSQL tables",
        "",
        "| API | " + " | ".join(all_tables) + " |",
        "|---|" + "---|" * len(all_tables),
    ]
    for api in sorted(matrix):
        lines.append(
            f"| `{api}` | " + " | ".join(matrix[api].get(t, "") or "·" for t in all_tables) + " |"
        )

    lines += ["", "## API → MinIO objects", ""]
    if rows:
        lines += ["| API | Method | Path | Count |", "|---|---|---|---|"]
        lines += [f"| `{r['api']}` | {r['method']} | `{r['path']}` | {r['count']} |" for r in rows]
    else:
        lines.append("_No MinIO activity captured in this run._")

    lines += ["", "## Per-API detail", ""]
    for rec in records:
        lines += [
            f"### `{rec.api}`",
            "",
            f"- `{rec.method} {rec.path}` → **{rec.status}**",
            f"- wall {rec.wall_ms:.0f} ms · {rec.sql_count} statements · "
            f"{rec.minio_count} object ops · cache: {rec.cache_verdict}",
            f"- tables: {', '.join(rec.tables_touched) or '—'}",
            "",
        ]
        if rec.sql:
            lines += ["| # | Table | Verb | ms | SQL |", "|---|---|---|---|---|"]
            for s in rec.sql:
                ms = f"{s.duration_ms:.2f}" if s.duration_ms is not None else "—"
                sql = s.sql.replace("|", "\\|")[:180]
                lines.append(f"| {s.seq} | {s.table or '—'} | {s.verb or '—'} | {ms} | `{sql}` |")
            lines.append("")
        if rec.minio:
            lines += ["| # | Method | Path | ms |", "|---|---|---|---|"]
            for m in rec.minio:
                ms = f"{m.duration_ms:.2f}" if m.duration_ms is not None else "—"
                lines.append(f"| {m.seq} | {m.method} | `{m.path}` | {ms} |")
            lines.append("")

    pathlib.Path("doc-api-sql-matrix.md").write_text("\n".join(lines), encoding="utf-8")

    a = [
        "# Index Audit",
        "",
        f"Generated {ts}. Schema version {drift['version_found']}, verdict "
        f"**{drift['verdict']}**.",
        "",
        "## Schema drift",
        "",
    ]
    a += [f"- {n}" for n in drift["notes"]]
    if drift["missing_indexes"]:
        a += ["", "### Missing expected access paths", ""]
        a += [
            f"- `{i['name']}` on `{i['table']}({', '.join(i['columns'])})` [{i['kind']}]"
            for i in drift["missing_indexes"]
        ]

    a += ["", "## Hypotheses", ""]
    for h in hyps:
        a += [
            f"### {h['id']} — **{h['status']}** (severity {h['severity']})",
            "",
            f"- Source: `{h['source_method']}` on `{h['table']}`",
            f"- Claim: {h['claim']}",
            f"- Impact: {h['why_it_matters']}",
            f"- Remedy: `{h['remedy']}`",
            "",
        ]

    a += ["", "## Statement verdicts", "",
          "| Verdict | Table | Verb | Calls | Rows | ms | Detail |", "|---|---|---|---|---|---|---|"]
    for r in sorted(audit, key=lambda x: x["verdict"] or ""):
        ms = f"{r['plan_ms']:.2f}" if r.get("plan_ms") else "—"
        a.append(
            f"| {r['verdict']} | {r['table'] or '—'} | {r['verb'] or '—'} | "
            f"{r['calls']} | {r['table_rows']} | {ms} | {r['detail'][:140]} |"
        )

    pathlib.Path("doc-index-audit.md").write_text("\n".join(a), encoding="utf-8")
    print("wrote doc-api-sql-matrix.md and doc-index-audit.md")


write_reports()

## 9. Teardown and cleanup

Removes the probe fixture, optionally the seeded fixture, and disposes of the database connection
(required by the repo's notebook conventions — the local PgBouncer pool is small).

In [ ]:
pc.delete_catalog(PROBE_CAT, purge=True)
for name, call in (
    ("principal_role", lambda: pc.delete_principal_role(PROBE_PROLE)),
    ("principal", lambda: pc.delete_principal(PROBE_PRINCIPAL)),
):
    call()
print(f"probe fixture {PROBE_CAT} removed")

strays = find_strays(pc, prefix=PROBE)
assert not any(strays.values()), f"probe entities left behind: {strays}"
print("no probe residue")

In [ ]:
TEARDOWN_SEED = False     # <-- set True to delete the ~16k-entity fixture

if TEARDOWN_SEED:
    def on_progress(done, total, res):
        print(f"  torn down {done}/{total}  elapsed={res.elapsed_s:.0f}s")

    tr = teardown(pc, ledger_path=LEDGER, on_progress=on_progress)
    print(tr.summary())
    if tr.failed_users:
        print(f"FAILED (re-run this cell): {tr.failed_users[:5]}")
    else:
        print("seed fixture fully removed; ledger cleared")
elif RUN_SEED:
    print(
        "Seed fixture LEFT IN PLACE — set TEARDOWN_SEED = True when finished.\n"
        f"Ledger: {LEDGER}"
    )

In [ ]:
conn.close()
for name in ("df_summary", "df_matrix", "df_audit", "df_idx"):
    if name in dir():
        del globals()[name]
gc.collect()
print("connections closed, frames released")

---

### Reverting the instrumentation

Run the revert section of `doc-instrumentation-runbook.md`. Leaving `log_statement='all'` enabled
will fill the disk and slow every subsequent run.

### Next phases

Deferred by decision (see the plan doc §13): the read-API cache/repeat protocol, the all-CRUD
performance approaches (`/q/metrics` per-API timers, OpenTelemetry span trees, write-path variable
sweeps), and the perspectives beyond DB/SQL/Index. If the grantee-index hypothesis came back
**CONFIRMED**, the follow-up is: create the index locally, re-run this notebook, record the
before/after delta, then check Polaris `main` before filing anything upstream.